# Prognoser Runner — Parameterized Longitudinal Survival Model

Edit the `EXPERIMENT` config cell, then run all cells.

**Methods:**
- `km` — Kaplan-Meier population baseline
- `cox_clinical` — Cox PH, clinical features only (baseline M0)
- `cox_embedding` — Cox PH, GAAE embedding (strategy-selected)
- `cox_combined` — Cox PH, clinical + GAAE embedding
- `cox_clinical_longitudinal` — Cox PH, clinical baseline + longitudinal aggregates
- `cox_time_varying` — Time-varying Cox (long-format per-visit data, symmetric)
- `rsf` — Random Survival Forest
- `deepsurv` — DeepSurv neural Cox (requires pycox)
- `lstm_surv` — LSTM discrete-time hazard model

**Embedding strategies** (for embedding-based methods):
- `baseline` — M0 only
- `last` — latest visit in at-risk window
- `mean` — mean across all window visits
- `slope` — linear trend (last − baseline)
- `all_aggs` — concat of all four above (4×latent_dim)

**At-risk window**: visits with `month < duration` per subject, applied symmetrically
to converters (window_end = event month) and non-converters (window_end = last visit month).

In [1]:
# ── EXPERIMENT CONFIG (papermill parameters) ──────────────────────────────────
# Edit for interactive Jupyter use. run_experiment.py overrides EXPERIMENT, SEED,
# and the RUN_* / WANDB_ENABLED / OUTPUT_DIR vars below via papermill.
SEED = 42
EXPERIMENT = {
    "network_combo": "dmn_hippo",
    "data_version": "__fc_dmn-hippo_sch200-tian2_flat__",
    "file_suffix": "_dmn_hippo_correlation_matrix_z_transformed.npz",
    "method": "cox_clinical_longitudinal",
        # km | cox_clinical | cox_embedding | cox_combined |
        # cox_clinical_longitudinal | cox_time_varying |
        # rsf | deepsurv | lstm_surv
    "feature_set": "clinical_longitudinal",
        # clinical | embedding | clinical+embedding |
        # clinical_longitudinal | clinical_longitudinal+embedding | sequence
    "embedding_strategy": "last",
        # baseline | last | mean | slope | all_aggs | sequence
    "longitudinal_features": ["mmstot", "cdrglobal"],
    "longitudinal_aggs": ["baseline", "last", "slope", "delta"],
    "eval_times": [12, 24, 36, 48, 60, 72],
    "penalizer": 0.05,
    "pca_components": 16,
    "rsf_n_estimators": 200,
    "rsf_min_samples_leaf": 5,
    "lstm_n_time_bins": 12,
    "lstm_max_horizon_months": 72,
    "lstm_hidden_dim": 64,
    "lstm_epochs": 100,
    "random_state": 42,
}
# Runner-injected parameters (None => interactive Jupyter behaviour):
EXPERIMENT_ID = None
RUN_DIR = None
RUN_NAME = None
WANDB_ENABLED = True
OUTPUT_DIR = None
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
# Parameters
EXPERIMENT_ID = "km-baseline"
EXPERIMENT = {"network_combo": "dmn_hippo", "data_version": "__fc_dmn-hippo_sch200-tian2_flat__", "file_suffix": "_dmn_hippo_correlation_matrix_z_transformed.npz", "method": "km", "feature_set": "clinical_longitudinal", "embedding_strategy": "last", "longitudinal_features": ["mmstot", "cdrglobal"], "longitudinal_aggs": ["baseline", "last", "slope", "delta"], "eval_times": [12, 24, 36, 48, 60, 72], "penalizer": 0.05, "pca_components": 16, "rsf_n_estimators": 200, "rsf_min_samples_leaf": 5, "lstm_n_time_bins": 12, "lstm_max_horizon_months": 72, "lstm_hidden_dim": 64, "lstm_epochs": 100, "random_state": 42}
SEED = 42
WANDB_ENABLED = True
OUTPUT_DIR = "outputs/km-baseline"
RUN_DIR = "/mnt/e/fyassine/ad-early-detection/PROGNOSER/outputs/km-baseline/runs/kind-rain-2-d9655b0db-2026-06-21_01-42-36"
RUN_NAME = "kind-rain-2-d9655b0db-2026-06-21_01-42-36"


In [3]:
import sys, os, json
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from DATA.src.splitting.load_splits import splits_dir
from CLASSIFIER.common import tracking, provenance
from CLASSIFIER.common.seeding import set_seed

# Process-wide seeding (random / numpy / torch / PYTHONHASHSEED / cuDNN).
# SEED is the single source of truth: lock the per-model random_state to it so
# interactive edits to SEED also reseed RSF / DeepSurv / LSTM.
set_seed(SEED)
EXPERIMENT['random_state'] = SEED

COHORTS_CSV = REPO_ROOT / 'DATA' / 'DELCODE' / '__fc_wholebrain_sch200_flat__' / 'metadata' / 'cohorts.csv'
SPLITS_DIR = splits_dir('downstream')
EMBEDDINGS_CACHE = REPO_ROOT / 'PROGNOSER' / 'notebooks' / '_embeddings_cache_'
CHECKPOINT_ROOT = REPO_ROOT / 'PROGNOSER' / 'notebooks' / f"checkpoints_prognoser_{EXPERIMENT['network_combo']}"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT['network_combo']} | Method: {EXPERIMENT['method']} | seed={SEED}")
print(f"Feature set: {EXPERIMENT['feature_set']} | Embedding strategy: {EXPERIMENT['embedding_strategy']}")

# W&B run: on by default, no-op stub when disabled / no creds, never blocks.
# Survival logs to its own project, separate from the classifier.
os.environ.setdefault('WANDB_PROJECT', 'ad-early-detection-prognosis')
_wandb_exp = {
    'id': EXPERIMENT_ID or RUN_NAME or f"{EXPERIMENT['method']}-{EXPERIMENT['network_combo']}",
    'model': EXPERIMENT['method'],
    'mode': 'survival',
    'dataset': EXPERIMENT['network_combo'],
    'seed': EXPERIMENT['random_state'],
    'wandb': WANDB_ENABLED,
}
wandb_run = tracking.init_run(_wandb_exp, {**EXPERIMENT, 'RUN_NAME': RUN_NAME})

Experiment: dmn_hippo | Method: km | seed=42
Feature set: clinical_longitudinal | Embedding strategy: last


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: lakhalfrajyassine (lakhalfrajyassine-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: setting up run z0l2wr2u


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /mnt/e/fyassine/ad-early-detection/PROGNOSER/wandb/run-20260621_014244-z0l2wr2u
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run km-baseline


wandb: ⭐️ View project at https://wandb.ai/lakhalfrajyassine-technical-university-of-munich/ad-early-detection-prognosis


wandb: 🚀 View run at https://wandb.ai/lakhalfrajyassine-technical-university-of-munich/ad-early-detection-prognosis/runs/z0l2wr2u


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PROGNOSER.common.survival_table import build_survival_table, make_xte
from PROGNOSER.common.metrics import evaluate_model
from PROGNOSER.common.longitudinal import to_long_format, to_sequence_tensors
from PROGNOSER.model.kaplan_meier import KMBaseline
from PROGNOSER.model.cox import CoxPHWrapper
from PROGNOSER.model.cox_time_varying import CoxTimeVaryingWrapper
from PROGNOSER.model.rsf import RSFWrapper
from PROGNOSER.model.lstm_surv import LSTMSurvWrapper

sns.set_theme(style='whitegrid')
print('Imports OK')

Imports OK


## Build Survival Tables (with at-risk window)

In [5]:
long_features = EXPERIMENT['longitudinal_features']
long_aggs = EXPERIMENT['longitudinal_aggs']

train_table = build_survival_table(
    str(COHORTS_CSV), splits_dir=str(SPLITS_DIR), split='train',
    longitudinal_features=long_features, longitudinal_aggs=long_aggs,
)
val_table = build_survival_table(
    str(COHORTS_CSV), splits_dir=str(SPLITS_DIR), split='val',
    longitudinal_features=long_features, longitudinal_aggs=long_aggs,
)
test_table = build_survival_table(
    str(COHORTS_CSV), splits_dir=str(SPLITS_DIR), split='test',
    longitudinal_features=long_features, longitudinal_aggs=long_aggs,
)

# At-risk windows from train (used for embedding extraction)
at_risk_windows_train = {str(row['subject_id']): int(row['duration']) for _, row in train_table.iterrows()}

for name, t in [('train', train_table), ('val', val_table), ('test', test_table)]:
    print(f'{name}: n={len(t):3d}  events={int(t.event_observed.sum()):3d}  '
          f'rate={t.event_observed.mean():.0%}  median_T={t.duration.median():.0f}m  '
          f'n_visits_median={t.n_visits_in_window.median():.1f}')

train: n= 92  events= 33  rate=36%  median_T=24m  n_visits_median=2.0
val: n= 33  events= 13  rate=39%  median_T=36m  n_visits_median=3.0
test: n= 32  events= 12  rate=38%  median_T=24m  n_visits_median=2.0


## Load GAAE Embeddings (if needed)

In [6]:
needs_embeddings = EXPERIMENT['method'] in ('cox_embedding', 'cox_combined', 'rsf', 'deepsurv')
embeddings_df = None

if needs_embeddings:
    strategy = EXPERIMENT['embedding_strategy']
    cache_file = EMBEDDINGS_CACHE / f"{EXPERIMENT['network_combo']}_{strategy}_embeddings.parquet"
    if not cache_file.exists():
        raise FileNotFoundError(
            f'Embeddings cache not found: {cache_file}\n'
            f'Run: python -m PROGNOSER.src.build_subject_embeddings '
            f'--combo {EXPERIMENT["network_combo"]} --strategy {strategy}'
        )
    embeddings_df = pd.read_parquet(cache_file)
    print(f'Loaded embeddings [{strategy}]: shape={embeddings_df.shape}')

    def _merge(table):
        return table.merge(embeddings_df, left_on='subject_id', right_index=True, how='inner').reset_index(drop=True)
    train_table = _merge(train_table)
    val_table   = _merge(val_table)
    test_table  = _merge(test_table)
    print(f'After merge: train={len(train_table)}, val={len(val_table)}, test={len(test_table)}')
else:
    print('Method does not require cached embeddings.')

Method does not require cached embeddings.


## Build Feature Matrices

In [7]:
clinical_cols = ['age', 'sex', 'mmstot', 'cdrglobal', 'apoe4']
embedding_cols = [c for c in (embeddings_df.columns if embeddings_df is not None else []) if c.startswith('z_')]
longitudinal_cols = [
    f'{feat}_{agg}' for feat in long_features for agg in long_aggs
    if f'{feat}_{agg}' in train_table.columns
]

if EXPERIMENT['feature_set'] == 'clinical':
    feature_cols = clinical_cols
elif EXPERIMENT['feature_set'] == 'embedding':
    feature_cols = embedding_cols
elif EXPERIMENT['feature_set'] == 'clinical+embedding':
    feature_cols = clinical_cols + embedding_cols
elif EXPERIMENT['feature_set'] == 'clinical_longitudinal':
    feature_cols = clinical_cols + longitudinal_cols
elif EXPERIMENT['feature_set'] == 'clinical_longitudinal+embedding':
    feature_cols = clinical_cols + longitudinal_cols + embedding_cols
elif EXPERIMENT['feature_set'] in ('sequence', 'cox_time_varying'):
    feature_cols = clinical_cols + longitudinal_cols  # used for long-format / sequence tensors
else:
    raise ValueError(f"Unknown feature_set: {EXPERIMENT['feature_set']}")

print(f'feature_cols ({len(feature_cols)}): {feature_cols[:8]}{"..." if len(feature_cols)>8 else ""}')

if EXPERIMENT['method'] not in ('cox_time_varying', 'lstm_surv'):
    X_tr, T_tr, E_tr, used_tr = make_xte(train_table, feature_cols)
    X_va, T_va, E_va, used_va = make_xte(val_table, feature_cols)
    X_te, T_te, E_te, used_te = make_xte(test_table, feature_cols)
    print(f'X_train: {X_tr.shape}  events: {E_tr.sum()}/{len(E_tr)}')
    print(f'X_val:   {X_va.shape}  events: {E_va.sum()}/{len(E_va)}')
    print(f'X_test:  {X_te.shape}  events: {E_te.sum()}/{len(E_te)}')

feature_cols (13): ['age', 'sex', 'mmstot', 'cdrglobal', 'apoe4', 'mmstot_baseline', 'mmstot_last', 'mmstot_slope']...
X_train: (53, 13)  events: 19/53
X_val:   (23, 13)  events: 10/23
X_test:  (18, 13)  events: 7/18


## Instantiate & Fit Model

In [8]:
method = EXPERIMENT['method']
cohorts_df = pd.read_csv(str(COHORTS_CSV))

if method == 'km':
    model = KMBaseline().fit(X_tr, T_tr, E_tr)

elif method == 'cox_clinical':
    model = CoxPHWrapper.with_clinical_features(penalizer=EXPERIMENT['penalizer']).fit(X_tr, T_tr, E_tr)

elif method == 'cox_clinical_longitudinal':
    model = CoxPHWrapper(feature_columns=feature_cols, penalizer=EXPERIMENT['penalizer']).fit(X_tr, T_tr, E_tr)

elif method == 'cox_embedding':
    latent_dim = len(embedding_cols)
    model = CoxPHWrapper.with_embedding_features(
        latent_dim=latent_dim,
        use_pca=True,
        pca_components=min(EXPERIMENT['pca_components'], latent_dim, X_tr.shape[0] - 1),
        penalizer=EXPERIMENT['penalizer'],
    ).fit(X_tr, T_tr, E_tr)

elif method == 'cox_combined':
    latent_dim = len(embedding_cols)
    model = CoxPHWrapper.with_clinical_plus_embedding(
        latent_dim=latent_dim,
        use_pca=True,
        pca_components=min(EXPERIMENT['pca_components'], X_tr.shape[1], X_tr.shape[0] - 1),
        penalizer=EXPERIMENT['penalizer'],
    ).fit(X_tr, T_tr, E_tr)

elif method == 'cox_time_varying':
    # Build long-format data for each split
    df_long_tr = to_long_format(train_table, cohorts_df, feature_cols)
    model = CoxTimeVaryingWrapper(feature_columns=feature_cols, penalizer=EXPERIMENT['penalizer'])
    model.fit_long(df_long_tr)
    # For evaluation we use last-visit features from the wide table
    X_tr, T_tr, E_tr, used_tr = make_xte(train_table, feature_cols)
    X_va, T_va, E_va, used_va = make_xte(val_table, feature_cols)
    X_te, T_te, E_te, used_te = make_xte(test_table, feature_cols)

elif method == 'rsf':
    model = RSFWrapper(
        feature_columns=feature_cols,
        n_estimators=EXPERIMENT['rsf_n_estimators'],
        min_samples_leaf=EXPERIMENT['rsf_min_samples_leaf'],
        random_state=EXPERIMENT['random_state'],
    ).fit(X_tr, T_tr, E_tr)

elif method == 'deepsurv':
    from PROGNOSER.model.deepsurv import DeepSurvWrapper
    model = DeepSurvWrapper(feature_columns=feature_cols, random_state=EXPERIMENT['random_state']).fit(
        X_tr, T_tr, E_tr, X_val=X_va, T_val=T_va, E_val=E_va
    )

elif method == 'lstm_surv':
    max_len = int(train_table['n_visits_in_window'].max()) + 1
    seq_tr, len_tr, T_tr, E_tr, ids_tr = to_sequence_tensors(train_table, cohorts_df, feature_cols, max_len)
    seq_va, len_va, T_va, E_va, ids_va = to_sequence_tensors(val_table, cohorts_df, feature_cols, max_len)
    seq_te, len_te, T_te, E_te, ids_te = to_sequence_tensors(test_table, cohorts_df, feature_cols, max_len)
    model = LSTMSurvWrapper(
        feature_columns=feature_cols,
        n_time_bins=EXPERIMENT['lstm_n_time_bins'],
        max_horizon_months=EXPERIMENT['lstm_max_horizon_months'],
        hidden_dim=EXPERIMENT['lstm_hidden_dim'],
        epochs=EXPERIMENT['lstm_epochs'],
        random_state=EXPERIMENT['random_state'],
    )
    model.fit_sequences(
        seq_tr, len_tr, T_tr, E_tr,
        val_data=(seq_va, len_va, T_va, E_va),
        epoch_callback=lambda e, tr, vl: tracking.log_metrics(
            wandb_run, {'epoch': e, 'train_loss': tr, 'val_loss': vl}),
    )
    # Wide-format proxies for evaluation (single-visit fallback)
    X_tr = seq_tr[:, 0, :]  # first visit features as proxy
    X_va = seq_va[:, 0, :]
    X_te = seq_te[:, 0, :]

else:
    raise ValueError(f'Unknown method: {method}')

print(f'Fitted: {model.method_name}')

Fitted: kaplan_meier


## Evaluate

In [9]:
eval_times = np.array(EXPERIMENT['eval_times'], dtype=float)

if method == 'lstm_surv':
    # Use sequence-aware predict methods
    from PROGNOSER.common.metrics import c_index, integrated_brier_score, time_dependent_auc
    def lstm_metrics(model, seq, lengths, T, E, T_ref, E_ref):
        risk = model.predict_risk_sequences(seq, lengths)
        surv = model.predict_survival_sequences(seq, lengths, eval_times)
        return {
            'c_index': c_index(T, E, risk),
            'ibs': integrated_brier_score(T_ref, E_ref, T, E, surv, eval_times),
            'auc': time_dependent_auc(T_ref, E_ref, T, E, risk, times=(24, 36, 60)),
        }
    metrics = {
        'train': lstm_metrics(model, seq_tr, len_tr, T_tr, E_tr, T_tr, E_tr),
        'val':   lstm_metrics(model, seq_va, len_va, T_va, E_va, T_tr, E_tr),
        'test':  lstm_metrics(model, seq_te, len_te, T_te, E_te, T_tr, E_tr),
    }
else:
    metrics = {
        'train': evaluate_model(model, X_tr, T_tr, E_tr, X_tr, T_tr, E_tr, eval_times=eval_times),
        'val':   evaluate_model(model, X_tr, T_tr, E_tr, X_va, T_va, E_va, eval_times=eval_times),
        'test':  evaluate_model(model, X_tr, T_tr, E_tr, X_te, T_te, E_te, eval_times=eval_times),
    }

print('=' * 65)
print(f'{"split":<6} {"C-index":>10} {"IBS":>10} {"AUC@24":>8} {"AUC@36":>8} {"AUC@60":>8}')
print('-' * 65)
for split_name, m in metrics.items():
    auc = m.get('auc', {})
    print(
        f'{split_name:<6} '
        f'{m["c_index"]:>10.4f} '
        f'{m["ibs"]:>10.4f} '
        f'{auc.get(24, float("nan")):>8.3f} '
        f'{auc.get(36, float("nan")):>8.3f} '
        f'{auc.get(60, float("nan")):>8.3f}'
    )
print('=' * 65)

split     C-index        IBS   AUC@24   AUC@36   AUC@60
-----------------------------------------------------------------
train      0.5000     0.2835      nan    0.500    0.500
val        0.5000     0.3714      nan    0.500    0.500
test       0.5000     0.3185      nan    0.500    0.500


## Save Results

In [10]:
if RUN_DIR:
    run_dir = Path(RUN_DIR)
    run_name = RUN_NAME or run_dir.name
    run_timestamp = run_dir.name
else:
    run_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    run_name = f"{method}_{EXPERIMENT['feature_set'].replace('+','-')}_{EXPERIMENT['embedding_strategy']}_{run_timestamp}"
    run_dir = CHECKPOINT_ROOT / run_name
run_dir.mkdir(parents=True, exist_ok=True)

if method != 'km':
    model.save(str(run_dir / f'model_{run_name}.joblib'))

# Test risk predictions
risk_te = model.predict_risk_sequences(seq_te, len_te) if method == 'lstm_surv' else model.predict_risk(X_te)
n_test = len(T_te)
test_subject_ids = ids_te if method == 'lstm_surv' else used_te['subject_id'].tolist()
T_te_save, E_te_save = T_te if method == 'lstm_surv' else (used_te['duration'].values, used_te['event_observed'].values)

preds_df = pd.DataFrame({'subject_id': test_subject_ids, 'duration': T_te_save, 'event_observed': E_te_save, 'risk': risk_te})
preds_df.to_csv(run_dir / 'predictions_test.csv', index=False)

run_summary = {
    'run_name': run_name, 'timestamp': run_timestamp,
    'experiment_id': EXPERIMENT_ID or run_name,
    'experiment': EXPERIMENT, 'method': method,
    'feature_set': EXPERIMENT['feature_set'],
    'embedding_strategy': EXPERIMENT['embedding_strategy'],
    'n_features': len(feature_cols),
    'feature_columns': feature_cols,
    'n_train': int(len(T_tr)), 'n_val': int(len(T_va)), 'n_test': int(n_test),
    'n_events_test': int(E_te.sum() if hasattr(E_te, 'sum') else sum(E_te)),
    'metrics': metrics,
    'eval_times': EXPERIMENT['eval_times'],
    'git': provenance.capture_git_provenance(),
    'env': provenance.capture_env(),
}
with open(run_dir / 'run_summary.json', 'w') as f:
    json.dump(run_summary, f, indent=2, default=str)

# Log final metrics to W&B (flattened) and close the run.
_wb_metrics = {}
for _split, _m in metrics.items():
    _wb_metrics[f'{_split}/c_index'] = _m.get('c_index')
    _wb_metrics[f'{_split}/ibs'] = _m.get('ibs')
    for _t, _v in (_m.get('auc') or {}).items():
        _wb_metrics[f'{_split}/auc_{_t}'] = _v
tracking.log_metrics(wandb_run, _wb_metrics)
tracking.finish_run(wandb_run)

print(f'Saved: {run_dir}')
print(f'Test C-index: {metrics["test"]["c_index"]:.4f}')

wandb: updating run metadata


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt


wandb: uploading history steps 0-0, summary, console lines 1-16


wandb: 
wandb: Run history:
wandb:   test/auc_36 ▁
wandb:   test/auc_60 ▁
wandb:  test/c_index ▁
wandb:      test/ibs ▁
wandb:  train/auc_36 ▁
wandb:  train/auc_60 ▁
wandb: train/c_index ▁
wandb:     train/ibs ▁
wandb:    val/auc_36 ▁
wandb:    val/auc_60 ▁
wandb:            +5 ...
wandb: 
wandb: Run summary:
wandb:   test/auc_24 nan
wandb:   test/auc_36 0.5
wandb:   test/auc_60 0.5
wandb:  test/c_index 0.5
wandb:      test/ibs 0.31847
wandb:  train/auc_24 nan
wandb:  train/auc_36 0.5
wandb:  train/auc_60 0.5
wandb: train/c_index 0.5
wandb:     train/ibs 0.28347
wandb:            +5 ...
wandb: 


wandb: 🚀 View run km-baseline at: https://wandb.ai/lakhalfrajyassine-technical-university-of-munich/ad-early-detection-prognosis/runs/z0l2wr2u
wandb: ⭐️ View project at: https://wandb.ai/lakhalfrajyassine-technical-university-of-munich/ad-early-detection-prognosis
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260621_014244-z0l2wr2u/logs


Saved: /mnt/e/fyassine/ad-early-detection/PROGNOSER/outputs/km-baseline/runs/kind-rain-2-d9655b0db-2026-06-21_01-42-36
Test C-index: 0.5000
